# Transcript-Level Benchmarking for Baleen (Real Data)

This notebook evaluates the pipeline's ability to identify **modified sites** (positions) within transcripts using real E. coli rRNA data with known modification sites.

## Key Metrics
- **Site-level ROC-AUC / PR-AUC**: Discrimination between modified and unmodified positions
- **V1 → V2 → V3 comparison**: Performance improvement across pipeline stages
- **Per-modification-type analysis**: Which modification types are detected best

In [ ]:
import sys
from pathlib import Path

if Path.cwd().name == 'notebooks':
    sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score, f1_score
import pandas as pd
import seaborn as sns
from collections import Counter

from baleen.eventalign import load_results
from baleen.eventalign._hierarchical import compute_sequential_modification_probabilities

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

# Find testdata
TESTDATA_PATHS = [
    Path.cwd().parent / 'testdata',
    Path.cwd() / 'testdata',
    Path('/Users/logan/Projects/py-baleen/testdata'),
]
TESTDATA_DIR = None
for p in TESTDATA_PATHS:
    if p.exists():
        TESTDATA_DIR = p
        break

if TESTDATA_DIR:
    print(f"Found testdata at: {TESTDATA_DIR}")
else:
    print("testdata not found - will use synthetic data for demonstration")

## 1. Known Modification Sites (E. coli 16S & 23S rRNA)

In [ ]:
POSITION_OFFSET = 3  # biological_pos = pipeline_pos + 3

KNOWN_MODIFICATIONS = {
    # --- Pseudouridine (Ψ) ---
    ("ecoli16S", 516):  ("Ψ", "pseudouridine"),
    ("ecoli23S", 746):  ("Ψ", "pseudouridine"),
    ("ecoli23S", 955):  ("Ψ", "pseudouridine"),
    ("ecoli23S", 1911): ("Ψ", "pseudouridine"),
    ("ecoli23S", 1917): ("Ψ", "pseudouridine"),
    ("ecoli23S", 2457): ("Ψ", "pseudouridine"),
    ("ecoli23S", 2504): ("Ψ", "pseudouridine"),
    ("ecoli23S", 2580): ("Ψ", "pseudouridine"),
    ("ecoli23S", 2604): ("Ψ", "pseudouridine"),
    ("ecoli23S", 2605): ("Ψ", "pseudouridine"),
    # --- N2-methylguanosine (m2G) ---
    ("ecoli16S", 966):  ("m2G", "N2-methylguanosine"),
    ("ecoli16S", 1207): ("m2G", "N2-methylguanosine"),
    ("ecoli16S", 1516): ("m2G", "N2-methylguanosine"),
    ("ecoli23S", 1835): ("m2G", "N2-methylguanosine"),
    ("ecoli23S", 2445): ("m2G", "N2-methylguanosine"),
    # --- 5-methylcytidine (m5C) ---
    ("ecoli16S", 967):  ("m5C", "5-methylcytidine"),
    ("ecoli16S", 1407): ("m5C", "5-methylcytidine"),
    ("ecoli23S", 1962): ("m5C", "5-methylcytidine"),
    # --- 5-methyluridine (m5U) ---
    ("ecoli23S", 747):  ("m5U", "5-methyluridine"),
    ("ecoli23S", 1939): ("m5U", "5-methyluridine"),
    # --- N6,N6-dimethyladenosine (m6,6A) ---
    ("ecoli16S", 1518): ("m6,6A", "N6,N6-dimethyladenosine"),
    ("ecoli16S", 1519): ("m6,6A", "N6,N6-dimethyladenosine"),
    # --- N6-methyladenosine (m6A) ---
    ("ecoli23S", 1618): ("m6A", "N6-methyladenosine"),
    ("ecoli23S", 2030): ("m6A", "N6-methyladenosine"),
    # --- N7-methylguanosine (m7G) ---
    ("ecoli16S", 527):  ("m7G", "N7-methylguanosine"),
    ("ecoli23S", 2069): ("m7G", "N7-methylguanosine"),
    # --- Other modifications ---
    ("ecoli23S", 2498): ("Cm", "2′-O-methyl cytosine"),
    ("ecoli23S", 2449): ("D", "dihydrouridine"),
    ("ecoli23S", 2251): ("Gm", "2′-O-methyl guanine"),
    ("ecoli23S", 2552): ("Um", "2′-O-methyl uridine"),
    ("ecoli23S", 2501): ("ho5C", "5-hydroxycytidine"),
    ("ecoli23S", 745):  ("m1G", "1-methylguanosine"),
    ("ecoli23S", 2503): ("m2A", "2-methyladenosine"),
    ("ecoli23S", 1915): ("m3Ψ", "3-methylpseudouridine"),
    ("ecoli16S", 1498): ("m3U", "3-methyluridine"),
    ("ecoli16S", 1402): ("m4Cm", "N4,2′-O-dimethylcytidine"),
}

KNOWN_MOD_PIPELINE = {
    (contig, bio_pos - POSITION_OFFSET): (mod_short, mod_full)
    for (contig, bio_pos), (mod_short, mod_full) in KNOWN_MODIFICATIONS.items()
}
KNOWN_MOD_SET = set(KNOWN_MOD_PIPELINE.keys())

mod_counts = Counter(v[0] for v in KNOWN_MODIFICATIONS.values())
print(f"Total known modification sites: {len(KNOWN_MODIFICATIONS)}")
print(f"Modification types: {len(mod_counts)}\n")
for mod_type, count in mod_counts.most_common():
    full = next(v[1] for v in KNOWN_MODIFICATIONS.values() if v[0] == mod_type)
    print(f"  {mod_type:6s} ({full}): {count} sites")

## 2. Load Pipeline Results

In [ ]:
RESULTS_PATHS = [
    Path.cwd().parent / 'output' / 'pipeline_results.pkl',
    Path.cwd().parent / 'results' / 'pipeline_results.pkl',
    Path.cwd() / 'output' / 'pipeline_results.pkl',
]

results = None
metadata = None

for p in RESULTS_PATHS:
    if p.exists():
        print(f"Loading results from: {p}")
        results, metadata = load_results(p)
        break

if results is None:
    print("No pre-computed results found.")
    print("\nRun the pipeline first:")
    print("  baleen run --native-bam testdata/native_0.bam \\")
    print("              --native-fastq testdata/native_0/fastq/pass.fq.gz \\")
    print("              --native-blow5 testdata/native_0/blow5/nanopore.blow5 \\")
    print("              --ivt-bam testdata/control_0.bam \\")
    print("              --ivt-fastq testdata/control_0/fastq/pass.fq.gz \\")
    print("              --ivt-blow5 testdata/control_0/blow5/nanopore.blow5 \\")
    print("              --ref testdata/ref.fa \\")
    print("              --output-dir output/")
    USE_SYNTHETIC = True
else:
    USE_SYNTHETIC = False
    print(f"\nLoaded {len(results)} contigs:")
    for contig, cr in results.items():
        print(f"  {contig}: {len(cr.positions)} positions")

## 3. Run Hierarchical Pipeline (V1 → V2 → V3)

In [ ]:
if not USE_SYNTHETIC and results is not None:
    hierarchical_results = {}
    
    for contig, cr in results.items():
        print(f"Processing {contig}...")
        hierarchical_results[contig] = compute_sequential_modification_probabilities(cr)
    
    print(f"\nCompleted hierarchical pipeline for {len(hierarchical_results)} contigs")

## 4. Extract Site-Level Scores

In [ ]:
def extract_site_level_scores(hierarchical_results, known_mod_set):
    """Extract per-position scores and ground truth."""
    data = {
        'contig': [], 'position': [], 'y_true': [],
        'mean_z_score': [], 'mean_p_mod_raw': [],
        'mean_p_mod_knn': [], 'mean_p_mod_hmm': [],
        'gate_weight': [], 'coverage_class': [], 'mod_type': [],
    }
    
    for contig, result in hierarchical_results.items():
        for pos, ps in result.position_stats.items():
            is_mod = (contig, pos) in known_mod_set
            mod_info = KNOWN_MOD_PIPELINE.get((contig, pos), ("unmod", "unmodified"))
            
            data['contig'].append(contig)
            data['position'].append(pos)
            data['y_true'].append(1 if is_mod else 0)
            data['mean_z_score'].append(np.mean(ps.native_z_scores))
            data['mean_p_mod_raw'].append(np.mean(ps.native_p_mod_raw))
            data['mean_p_mod_knn'].append(np.nanmean(ps.native_p_mod_knn))
            data['mean_p_mod_hmm'].append(np.nanmean(ps.native_p_mod_hmm))
            data['gate_weight'].append(ps.gate_weight)
            data['coverage_class'].append(ps.coverage_class.value)
            data['mod_type'].append(mod_info[0])
    
    return pd.DataFrame(data)


if not USE_SYNTHETIC and 'hierarchical_results' in dir():
    site_df = extract_site_level_scores(hierarchical_results, KNOWN_MOD_SET)
    print(f"Extracted {len(site_df)} site-level predictions")
    print(f"  Known modification sites: {site_df['y_true'].sum()}")
    print(f"  Unmodified sites: {(site_df['y_true'] == 0).sum()}")
    site_df.head(10)

## 5. ROC Curves by Pipeline Stage

In [ ]:
if not USE_SYNTHETIC and 'site_df' in dir() and len(site_df) > 0:
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    
    score_cols = ['mean_z_score', 'mean_p_mod_raw', 'mean_p_mod_knn', 'mean_p_mod_hmm']
    labels = ['V1: Z-Score', 'V2: P(mod) Raw', 'V2: P(mod) kNN', 'V3: P(mod) HMM']
    colors = ['#9C27B0', '#4CAF50', '#2196F3', '#FF5722']
    
    y_true = site_df['y_true'].values
    
    for ax, col, label, color in zip(axes, score_cols, labels, colors):
        scores = site_df[col].values
        valid = ~np.isnan(scores)
        
        if len(np.unique(y_true[valid])) < 2:
            ax.text(0.5, 0.5, 'Insufficient data', ha='center', va='center')
            continue
        
        fpr, tpr, _ = roc_curve(y_true[valid], scores[valid])
        roc_auc = auc(fpr, tpr)
        pr_auc = average_precision_score(y_true[valid], scores[valid])
        
        ax.plot(fpr, tpr, '-', color=color, lw=2, label=f'ROC (AUC={roc_auc:.3f})')
        ax.plot([0, 1], [0, 1], 'k--', lw=1)
        ax.set_xlabel('False Positive Rate')
        ax.set_ylabel('True Positive Rate')
        ax.set_title(f'{label}\nPR-AUC={pr_auc:.3f}')
        ax.legend(loc='lower right')
        ax.set_xlim([-0.02, 1.02])
        ax.set_ylim([-0.02, 1.02])
    
    plt.suptitle('Site-Level ROC Curves by Pipeline Stage (Known Modifications as Ground Truth)', y=1.02)
    plt.tight_layout()
    plt.savefig('transcript_level_real_pipeline_stages.png', dpi=150, bbox_inches='tight')
    plt.show()

## 6. Performance by Modification Type

In [ ]:
if not USE_SYNTHETIC and 'site_df' in dir() and len(site_df) > 0:
    mod_metrics = []
    
    for mod_type in site_df['mod_type'].unique():
        if mod_type == 'unmod':
            continue
        
        subset = site_df[site_df['mod_type'] == mod_type]
        if len(subset) < 2:
            continue
        
        # For site-level, we compare modified vs all unmodified
        y_true_binary = (site_df['mod_type'] == mod_type).astype(int).values
        
        for col, stage in [('mean_z_score', 'V1'), ('mean_p_mod_knn', 'V2_kNN'), ('mean_p_mod_hmm', 'V3_HMM')]:
            scores = site_df[col].values
            valid = ~np.isnan(scores)
            
            if len(np.unique(y_true_binary[valid])) < 2:
                continue
            
            try:
                roc = auc(*roc_curve(y_true_binary[valid], scores[valid])[:2])
            except:
                roc = np.nan
            
            mod_metrics.append({
                'mod_type': mod_type,
                'stage': stage,
                'roc_auc': roc,
                'n_sites': len(subset),
            })
    
    if mod_metrics:
        mod_df = pd.DataFrame(mod_metrics)
        
        fig, ax = plt.subplots(figsize=(12, 5))
        pivot = mod_df.pivot(index='mod_type', columns='stage', values='roc_auc')
        pivot = pivot.reindex(columns=['V1', 'V2_kNN', 'V3_HMM'])
        pivot.plot(kind='bar', ax=ax, color=['#9C27B0', '#2196F3', '#FF5722'])
        ax.set_xlabel('Modification Type')
        ax.set_ylabel('ROC-AUC')
        ax.set_title('Site-Level ROC-AUC by Modification Type (One-vs-Rest)')
        ax.legend(title='Pipeline Stage')
        ax.set_ylim([0, 1.05])
        ax.axhline(y=0.5, color='gray', ls='--', lw=1)
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.savefig('transcript_level_by_mod_type.png', dpi=150, bbox_inches='tight')
        plt.show()

## 7. Probability Tracks Along Transcript

In [ ]:
if not USE_SYNTHETIC and 'site_df' in dir() and len(site_df) > 0:
    # Plot for each contig
    for contig in site_df['contig'].unique():
        subset = site_df[site_df['contig'] == contig].sort_values('position')
        
        if len(subset) < 10:
            continue
        
        fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
        
        positions = subset['position'].values
        
        # Top: probability tracks
        ax = axes[0]
        ax.fill_between(positions, 0, subset['y_true'].values, alpha=0.2, color='red', label='Known Modified')
        ax.plot(positions, subset['mean_p_mod_knn'], 'o-', color='#2196F3',
                markersize=4, lw=1, alpha=0.7, label='V2: kNN')
        ax.plot(positions, subset['mean_p_mod_hmm'], 's-', color='#FF5722',
                markersize=4, lw=1, alpha=0.7, label='V3: HMM')
        
        ax.set_ylabel('Mean P(modified)')
        ax.set_title(f'Probability Tracks - {contig}')
        ax.legend(loc='upper right')
        ax.set_ylim([-0.05, 1.05])
        
        # Bottom: z-scores
        ax = axes[1]
        colors = np.where(subset['y_true'], 'red', 'gray')
        ax.bar(positions, subset['mean_z_score'], color=colors, alpha=0.7, width=0.8)
        ax.axhline(y=0, color='k', lw=0.5)
        ax.axhline(y=2, color='red', ls='--', lw=1, alpha=0.5)
        
        ax.set_xlabel('Position')
        ax.set_ylabel('Mean Z-Score')
        ax.set_title('V1 Z-Scores')
        
        plt.tight_layout()
        plt.savefig(f'transcript_level_tracks_{contig}.png', dpi=150, bbox_inches='tight')
        plt.show()
        
        # Only show first contig
        break

## 8. Known Modification Site Details

In [ ]:
if not USE_SYNTHETIC and 'site_df' in dir() and len(site_df) > 0:
    # Show known modification sites with their scores
    known_sites = site_df[site_df['y_true'] == 1].copy()
    
    if len(known_sites) > 0:
        print(f"\nKnown Modification Sites Found in Data: {len(known_sites)}")
        print("=" * 80)
        
        display_cols = ['contig', 'position', 'mod_type', 'mean_z_score', 
                       'mean_p_mod_knn', 'mean_p_mod_hmm', 'coverage_class']
        print(known_sites[display_cols].round(3).to_string(index=False))
        
        # Summary by modification type
        print("\n" + "=" * 80)
        print("Mean Scores by Modification Type:")
        print("=" * 80)
        type_summary = known_sites.groupby('mod_type').agg({
            'mean_z_score': 'mean',
            'mean_p_mod_knn': 'mean',
            'mean_p_mod_hmm': 'mean',
            'position': 'count'
        }).round(3)
        type_summary.columns = ['Z-Score', 'P(kNN)', 'P(HMM)', 'N_Sites']
        print(type_summary.to_string())
    else:
        print("No known modification sites found in the data.")

## 9. Coverage Class Analysis

In [ ]:
if not USE_SYNTHETIC and 'site_df' in dir() and len(site_df) > 0:
    # Performance by IVT coverage class
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    for ax, (col, title) in zip(axes, [('mean_p_mod_knn', 'V2: kNN'), ('mean_p_mod_hmm', 'V3: HMM')]):
        for cc in ['high', 'medium', 'low']:
            cc_subset = site_df[site_df['coverage_class'] == cc]
            if len(cc_subset) == 0:
                continue
            
            mod_scores = cc_subset[cc_subset['y_true'] == 1][col].dropna()
            unmod_scores = cc_subset[cc_subset['y_true'] == 0][col].dropna()
            
            if len(mod_scores) > 0:
                ax.hist(unmod_scores, bins=20, alpha=0.5, label=f'{cc} unmod (n={len(unmod_scores)})',
                        density=True)
                ax.hist(mod_scores, bins=10, alpha=0.7, label=f'{cc} mod (n={len(mod_scores)})',
                        density=True)
        
        ax.set_xlabel('P(modified)')
        ax.set_ylabel('Density')
        ax.set_title(title)
        ax.legend(fontsize=8)
        ax.set_xlim([-0.02, 1.02])
    
    plt.suptitle('Score Distributions by Coverage Class', y=1.02)
    plt.tight_layout()
    plt.savefig('transcript_level_coverage_class.png', dpi=150, bbox_inches='tight')
    plt.show()

## 10. Summary Statistics

In [ ]:
if not USE_SYNTHETIC and 'site_df' in dir() and len(site_df) > 0:
    y_true = site_df['y_true'].values
    
    summary = []
    for col, stage in [('mean_z_score', 'V1: Z-Score'), 
                       ('mean_p_mod_raw', 'V2: P(mod) Raw'),
                       ('mean_p_mod_knn', 'V2: P(mod) kNN'), 
                       ('mean_p_mod_hmm', 'V3: P(mod) HMM')]:
        scores = site_df[col].values
        valid = ~np.isnan(scores)
        
        if len(np.unique(y_true[valid])) < 2:
            continue
        
        roc = auc(*roc_curve(y_true[valid], scores[valid])[:2])
        pr = average_precision_score(y_true[valid], scores[valid])
        
        # Best F1
        thresholds = np.linspace(scores[valid].min(), scores[valid].max(), 100)
        f1s = [f1_score(y_true[valid], scores[valid] > t, zero_division=0) for t in thresholds]
        best_f1 = max(f1s)
        
        summary.append({
            'Stage': stage,
            'ROC-AUC': roc,
            'PR-AUC': pr,
            'Best F1': best_f1,
        })
    
    summary_df = pd.DataFrame(summary)
    
    print("\n" + "="*70)
    print("TRANSCRIPT-LEVEL BENCHMARK SUMMARY (Real Data)")
    print("="*70)
    print(f"\nTotal sites: {len(site_df)}")
    print(f"Known modification sites: {site_df['y_true'].sum()}")
    print(f"Unmodified sites: {(site_df['y_true'] == 0).sum()}")
    print(f"\nContigs: {site_df['contig'].nunique()}")
    print(f"Modification types detected: {site_df[site_df['y_true'] == 1]['mod_type'].nunique()}")
    print()
    print(summary_df.round(3).to_string(index=False))
    
    # Save summary
    summary_df.to_csv('transcript_level_summary.csv', index=False)
    print(f"\nSummary saved to: transcript_level_summary.csv")